# Lyrics Transcribe in Colab

**Pipeline:** anvuew BS-Roformer (vocal isolation) → WhisperX or GigaAM (ASR with word timestamps) → standalone karaoke HTML page.

**Runtime:** GPU required — *Runtime → Change runtime type → T4 GPU* (or better).

**What you upload:** only the audio file(s) you want to transcribe. The code is cloned from GitHub, the anvuew checkpoint is fetched from a release, WhisperX (~3 GB from HuggingFace) and GigaAM (~430 MB from Sber CDN) auto-download into `models/` on first use.

## 1. Setup — clone, install, env, download anvuew checkpoint (~196 MB)

One-shot init: clones the repo, creates a `.venv` with torch+cuDNN and project deps, sets the env vars faster-whisper/gigaam need, and pulls the anvuew BS-Roformer checkpoint from the GitHub release (it's >100 MB so it lives in a release, not the repo).

WhisperX (~3 GB) and GigaAM (~430 MB) auto-download into `models/` on first ASR call. MMS aligner (~1.2 GB) auto-downloads on first `--lyrics` call.

In [ ]:
# 1) clone repo
!git clone https://github.com/Beatloo-Labs/LyricsTranscribe.git /content/LyricsTranscribe
%cd /content/LyricsTranscribe

# 2) venv + deps (torch from pytorch index for the CUDA build)
!pip install -q uv
!uv venv
!uv pip install -p .venv/bin/python --index-url https://download.pytorch.org/whl/cu128 torch==2.8.0 torchaudio==2.8.0
!uv pip install -p .venv/bin/python -r requirements.txt

# 3) env: cuDNN libs (from the nvidia-cudnn pip wheel) + system NVIDIA driver
import os, glob
cudnn = glob.glob('/content/LyricsTranscribe/.venv/lib/python*/site-packages/nvidia/cudnn/lib/')
if not cudnn:
    raise RuntimeError('cuDNN libs not found — torch install must have failed above')
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = 'true'
os.environ['LD_LIBRARY_PATH'] = cudnn[0] + ':/usr/lib64-nvidia:' + os.environ.get('LD_LIBRARY_PATH', '')

# 4) anvuew checkpoint
!mkdir -p models/anvuew
!wget -q --show-progress \
  https://github.com/Beatloo-Labs/LyricsTranscribe/releases/download/weights-v1/bs_roformer_ft1_anvuew_sdr_12.55.ckpt \
  -O models/anvuew/bs_roformer_ft1_anvuew_sdr_12.55.ckpt

## 2. Upload audio file(s) to transcribe

In [ ]:
from google.colab import files

# 1) Upload audio file(s) — mp3 / wav / flac / m4a / ogg
print('Upload AUDIO file(s):')
audio = files.upload()
AUDIO_FILES = [n for n in audio if not n.lower().endswith('.txt')]
print('audio:', AUDIO_FILES)

# 2) (optional) upload a .txt with the exact lyrics. If you upload one,
#    the next cell will switch to forced-alignment mode automatically:
#    words come from your text, timings from MMS — no ASR errors.
#    Format: one phrase per line, UTF-8. Skip this to run plain ASR.
print('\nUpload lyrics .txt (optional, cancel to skip):')
try:
    lyrics = files.upload()
    LYRICS_FILE = next((n for n in lyrics if n.lower().endswith('.txt')), None)
except Exception:
    LYRICS_FILE = None
print('lyrics:', LYRICS_FILE or '(none — will use ASR)')


## 3. Run the CLI

Pick `--model whisperx` (multilingual) or `--model gigaam` (Russian-tuned). First run downloads the chosen ASR model (~3 GB for whisperx, ~430 MB for gigaam) into `models/`. Each track lands in its own `cli-out/<name>__<model>__<timestamp>/` subfolder with the audio, JSON, and karaoke HTML.

In [ ]:
import subprocess

MODEL    = 'whisperx'   # 'whisperx' or 'gigaam' (ignored if LYRICS_FILE is set)
LANGUAGE = 'ru'         # ISO code, or empty string for auto-detect
ISOLATE  = True         # run anvuew vocal separation before ASR/align
OUT_DIR  = 'cli-out'

args = ['.venv/bin/python', 'transcribe.py',
        '--language', LANGUAGE, '--out', OUT_DIR]
if not ISOLATE:
    args.append('--no-isolate')

if LYRICS_FILE:
    # forced alignment: words from your .txt, timings from MMS
    # --model is ignored in this mode
    args += ['--lyrics', LYRICS_FILE]
    print(f'[align] using lyrics: {LYRICS_FILE}')
else:
    args += ['--model', MODEL]
    print(f'[asr] using model: {MODEL}')

args += AUDIO_FILES
subprocess.run(args, env=os.environ, check=True)


## 4. Download results

Each track produces three files in its `cli-out/<name>__<model>__<timestamp>/` subfolder: the original audio, `<name>.json` (microformat), and `<name>.html` (standalone karaoke page that auto-loads the sibling audio). Zip everything and download:

In [ ]:
!ls -la cli-out/
!cd cli-out && zip -rq ../karaoke-output.zip .
from google.colab import files
files.download('karaoke-output.zip')

## (Optional) Run the FastAPI server in Colab

Colab can't expose a port to the public web directly, but `serve_kernel_port_as_window` proxies it through the Colab UI.

In [ ]:
import subprocess, time
proc = subprocess.Popen(['.venv/bin/python', 'server.py'], env=os.environ)
time.sleep(4)
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8000)